In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import optuna
from sklearn.cluster import BisectingKMeans, AgglomerativeClustering
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import silhouette_score
import numpy as np
from scipy.cluster.hierarchy import dendrogram, linkage, cut_tree

## 1: Ler dados

In [ ]:
df_species = pd.read_csv("./dataset/species_dataset.csv")
df_species.info()

## 2: Filtrar Dados

Resultado: Não há dados faltantes, mas a coluna de ID é inútil para o agrupamento, já que não é um campo em comum entre quaisquer 2 registros. Logo, ela foi retirada.

In [ ]:
df_species.drop(columns=['species_id'], inplace=True)
df_species.info()

In [ ]:
df_species.describe()

In [ ]:
df_species['social_behavior'].unique()

In [ ]:
df_species['diet_type'].unique()

In [ ]:
df_species['skin_type'].unique()

## 3: Análise Exploratória

### Análise Univariada

Distribuição Peso

In [ ]:
sns.histplot(
  data=df_species,
  x='body_mass_kg',
  kde=True,
  color='darkcyan',
)
plt.title("Weight Distribution")
plt.ylabel("Frequency")
plt.xlabel("Weight (Kg)")
plt.grid(visible=True)

Distribuição tamanho da cauda

In [ ]:
sns.histplot(
  data=df_species,
  x='tail_length_cm',
  kde=True,
  color='darkred'
)
plt.title("Tail Length Distribution")
plt.ylabel("Frequency")
plt.xlabel("Tail Length (cm)")
plt.grid(visible=True)

Heatmap quantidade de olhos

In [ ]:
table_heatmap = df_species.value_counts(
  'eye_count', normalize=True,
  ).sort_index(ascending=True)
sns.barplot(
  table_heatmap, # type: ignore
  color='lightblue',
)
plt.grid(visible=True, alpha=0.5)
plt.ylabel("Percentage Frequency")
plt.xlabel("Number of Eyes")
plt.title("Eye Count Distribution")


Distribuição presença de asas

In [ ]:
table_heatmap = df_species.value_counts(
  'has_wings',
  normalize=True, # type: ignore 
  ).sort_index(ascending=True)
sns.barplot(
  table_heatmap, # type: ignore
  color='orange',
)
plt.grid(visible=True, alpha=0.5)
plt.ylabel("Percentage Frequency")
plt.xlabel("Has Wings (0=No, 1=Yes)")
plt.title("Wings Presence Distribution")

### Análise Bivariada

Peso por número de asas

In [ ]:
crosstable = pd.crosstab(
  df_species.has_wings,
  df_species.eye_count,
)
sns.heatmap(
  crosstable,
  annot=True,
  cmap='Reds',
  vmin=0,
  vmax=df_species.shape[0],
  fmt='d',
)
plt.xlabel("Number of Eyes")
plt.ylabel("Wings Presence")
plt.title("Number of Eyes x Wings Presence Heatmap")


Scatter Peso e Anos de Vida

In [ ]:
sns.scatterplot(
  data=df_species,
  x='body_mass_kg',
  y='avg_lifespan_years',
  hue='has_wings',
  palette=['darkred', 'darkcyan', 'darkgreen'],
)
plt.grid(visible=True, alpha=0.3)
plt.title("Average Lifetime (years) x Body Mass Scatterplot")
plt.ylabel("Average Lifetime (years)")
plt.xlabel('Body Mass (Kg)')

### Treinar modelo

Transformar dados

In [ ]:
X = df_species.copy()

numeric_features = [
  'body_mass_kg',
  'num_legs',
  'has_wings',
  'tail_length_cm',
  'eye_count',
  'nocturnal',
  'avg_lifespan_years',
  'has_venom',
]
categorical_features = [
  'diet_type',
  'skin_type',
  'social_behavior',
]

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder()

preprocessor = ColumnTransformer(transformers=[
  ('num', numeric_transformer, numeric_features),
  ('cat', categorical_transformer, categorical_features),
])

X_transformed = preprocessor.fit_transform(X)

X_transformed

Treinar modelo aglomerativo com optuna

In [ ]:
def agglomerative_objective(trial: optuna.Trial):
    linkg = trial.suggest_categorical(
      "linkage",
      ['ward', 'complete', 'average', 'single']
    )
    n_clusters = trial.suggest_int("n_cluster", 10, 200)

    model_opt_agg = AgglomerativeClustering(
      n_clusters=n_clusters,
      linkage=linkg # type: ignore
    )

    y = model_opt_agg.fit_predict(X_transformed)

    s_score = silhouette_score(X_transformed, y)

    return s_score

search_space = {
    'linkage': ['ward', 'complete', 'average', 'single'],
    'n_cluster': list(range(10, 200+1))
}
sampler = optuna.samplers.GridSampler(search_space=search_space)
opt_study_agg = optuna.create_study(
  sampler=sampler,
  study_name="Agglomerative Clustering Study",
  direction='maximize',
)
opt_study_agg.optimize(
  agglomerative_objective, # type: ignore
  n_trials=760,
  show_progress_bar=True
)

Treinar modelo divisivo com Optuna

In [ ]:
def divisive_objective(trial: optuna.Trial):
    n_clusters = trial.suggest_int("n_cluster", 10, 600)

    model_opt_agg = BisectingKMeans(
        n_clusters=n_clusters
    )

    y = model_opt_agg.fit_predict(X_transformed)

    s_score = silhouette_score(X_transformed, y)

    return s_score

search_space = {
    'n_cluster': list(range(10, 200+1))
}
sampler = optuna.samplers.GridSampler(search_space=search_space)
opt_study_div = optuna.create_study(
  sampler=sampler,
  study_name="Divisive Clustering Study",
  direction='maximize',
)
opt_study_div.optimize(
  divisive_objective, # type: ignore
  n_trials=190,
  show_progress_bar=True
)

### Analisar métricas do modelo
Como a métrica do modelo de agregação é melhor, seguiu-se com esse modelo


In [ ]:
print(f"Silhouette Score (Divisive): {opt_study_div.best_value}")
print(f"Number of Clusters (Divisive): {opt_study_div.best_params['n_cluster']}")
print(f"Silhouette Score (Agglomerative): {opt_study_agg.best_value}")
print(f"Number of Clusters (Agglomerative): {opt_study_agg.best_params['n_cluster']}")

Treinar modelo otimizado

In [ ]:
best_model = AgglomerativeClustering(
  n_clusters=opt_study_agg.best_params['n_cluster'],
  linkage=opt_study_agg.best_params['linkage']
)

best_model.fit(X_transformed)

df_species['cluster'] = best_model.labels_

s_score = silhouette_score(X_transformed, best_model.labels_)

s_score

### Dendrograma

Dendograma

In [ ]:
modelo_dendrograma = linkage(
  X_transformed,
  method=opt_study_agg.best_params['linkage'],
  optimal_ordering=True,
)
plt.figure(figsize=(18, 6))
dendrogram(
  modelo_dendrograma,
  truncate_mode='lastp',
  p=193,
  leaf_rotation=80,
  leaf_font_size=10,
)

Cortar dendograma

In [ ]:
altura=12.145
clusters_na_altura = cut_tree(modelo_dendrograma, height=altura)
print(f"Clusters na altura {altura}: {len(np.unique(clusters_na_altura))}")

### Análise do Modelo com Clusters

Peso x Tempo de Vida | Cluster

In [ ]:
sns.scatterplot(
  data=df_species,
  x='avg_lifespan_years',
  y='body_mass_kg',
  hue='cluster',
)
plt.grid(visible=True, alpha=0.3)
plt.title("Body Mass x Average Lifespan")
plt.ylabel("Body Mass (Kg)")
plt.xlabel("Average Lifetime (years)")

Distribuição dos Clusters

In [ ]:
cluster_values = df_species.value_counts(
  'cluster',
  normalize=True
).sort_index(ascending=True)
plt.figure(figsize=(18,6))
sns.barplot(
  x=cluster_values.index.values,
  y=cluster_values.values,
  hue=cluster_values.index.values,
  palette='magma',
  legend=False,
)
plt.xticks([])
plt.xlabel('Clusters (0 to 150)')
plt.ylabel("Percentage Frequency")
plt.title("Cluster Distribution")
cluster_values

### Salvar modelo

In [ ]:
joblib.dump(best_model, "./model_agg.pkl")